In [66]:
# 필요한 모듈 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import SGD, Adam

from sklearn.model_selection import cross_val_score

In [67]:
# 데이터 로딩
df = pd.read_csv('/content/drive/MyDrive/KDThome/train.csv')
#display(df.head(10))
df.info()
# 데이터가 부족하거나 불필요한 컬럼은 삭제(차원축소)
df = df.drop(['Cabin','Name','Ticket','Fare'], axis=1, inplace=False)
display(df.head())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,1,0,3,male,22.0,1,0,S
1,2,1,1,female,38.0,1,0,C
2,3,1,3,female,26.0,0,0,S
3,4,1,1,female,35.0,1,0,S
4,5,0,3,male,35.0,0,0,S


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Embarked     889 non-null    object 
dtypes: float64(1), int64(5), object(2)
memory usage: 55.8+ KB


In [68]:
# 데이터 전처리
# 1. 결측치확인
print(df.isnull().sum())
# 결측치가 나이에 많이 있음. 이건 평균으로
df = df.dropna(subset='Embarked')
df = df['Age'].fillna(df['Age'].median())

print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Embarked         2
dtype: int64
0


In [69]:
# 2. 이상치확인
# 이상치는 아니지만 문자를 숫자로 바꿔주자
df['Sex'] = df['Sex'].replace({'female':1, 'male':0})
df['Embarked'] = df['Embarked'].replace({'C':0, 'Q':1, 'S':2})

# 학습 데이터 분리
x_data = df.drop('Survived', axis=1, inplace=False).values
t_data = df['Survived'].values.reshape(-1,1)

# 3. 정규화처리
scaler = MinMaxScaler()
scaler.fit(x_data)
x_data_norm = scaler.transform(x_data)


KeyError: 'Sex'

In [54]:
# 모델구현
model = Sequential()
model.add(Flatten(input_shape=(6,)))
model.add(Dense(units=1,
                activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(x_data_norm,
          t_data,
          epochs=1000,
          verbose=1,
          validation_split=0.2,
          batch_size=100)

Epoch 1/1000


/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_5" is incompatible with the layer: expected axis -1 of input shape to have value 6, but received input with shape (None, 7)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 7), dtype=float32)
  • training=True
  • mask=None

In [ ]:
test_data = pd.read_csv('/content/drive/MyDrive/KDThome/test.csv')
scaler = MinMaxScaler()
X_test = scaler.fit_transform(test_data.values)
predictions = model.predict(X_test)  # 확률 값 나옴
# 이진 분류일 경우 → 확률 → 클래스(0 또는 1)로 변환
predictions = (predictions > 0.5).astype(int)
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': predictions.flatten()
})
submission.to_csv('submission.csv', index=False)

from google.colab import files
files.download('submission.csv')